# JACTUS Demo: CLI + Python for Financial Contract Simulation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/07_demo_cli_and_python.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-JACTUS-blue?logo=github)](https://github.com/pedronahum/JACTUS)
[![PyPI](https://img.shields.io/pypi/v/jactus)](https://pypi.org/project/jactus/)
[![License](https://img.shields.io/badge/License-Apache%202.0-blue.svg)](https://opensource.org/licenses/Apache-2.0)

**JACTUS** implements all 18 [ACTUS](https://www.actusfrf.org/) financial contract types using [JAX](https://jax.readthedocs.io/), with a powerful CLI and Python API.

This notebook demonstrates both interfaces side by side:

| Section | What You'll See |
|---------|----------------|
| **Part 1** | CLI -- discover, validate, simulate, and compute risk from the terminal |
| **Part 2** | Python API -- the same contracts with full programmatic control |
| **Part 3** | Behavioral Risk -- PrepaymentSurfaceObserver with 2D surface visualization |
| **Part 4** | Performance -- batch simulation of 1,000+ contracts with JAX |

In [ ]:
!pip install -q --upgrade jactus --no-deps

---
## Part 1: The JACTUS CLI

JACTUS ships with a full command-line interface. No Python required -- just `jactus` from the terminal.

```
jactus
  contract   list | schema | validate
  simulate   run a single contract
  risk       dv01 | duration | convexity | sensitivities
  portfolio  simulate | aggregate
  observer   list | describe
  docs       search
```

### 1.1 Discover All Contract Types

In [ ]:
!jactus contract list

### 1.2 Inspect a Contract Schema

Before building a contract, check what fields are required.

In [ ]:
!jactus contract schema --type PAM --required-only

### 1.3 Validate Before You Simulate

Catch errors early -- `jactus contract validate` checks attributes without running the simulation. Let's first save our contract to a file.

In [ ]:
%%writefile /tmp/bond.json
{
  "contract_id": "BOND-001",
  "contract_role": "RPA",
  "status_date": "2024-01-01",
  "initial_exchange_date": "2024-01-15",
  "maturity_date": "2027-01-15",
  "notional_principal": 100000,
  "nominal_interest_rate": 0.05,
  "interest_payment_cycle": "6M",
  "day_count_convention": "30E360"
}

In [ ]:
!jactus contract validate --type PAM --attrs /tmp/bond.json

### 1.4 Simulate a PAM Bond (CLI)

One command -- full cash flow schedule. We reuse the same `/tmp/bond.json` file.

In [ ]:
!jactus simulate --type PAM --attrs /tmp/bond.json

### 1.5 Simulate a Mortgage (ANN) via CLI

In [ ]:
%%writefile /tmp/mortgage.json
{
  "contract_id": "MORTGAGE-001",
  "contract_role": "RPA",
  "status_date": "2024-01-01",
  "initial_exchange_date": "2024-01-15",
  "maturity_date": "2029-01-15",
  "notional_principal": 200000,
  "nominal_interest_rate": 0.06,
  "interest_payment_cycle": "1M",
  "principal_redemption_cycle": "1M",
  "day_count_convention": "A360"
}

In [ ]:
!jactus simulate --type ANN --attrs /tmp/mortgage.json

### 1.6 Simulate a Futures Contract (CLI)

A gold futures contract with a market observer for the spot price.

In [ ]:
%%writefile /tmp/gold_future.json
{
  "contract_id": "GOLD-FUT",
  "contract_role": "BUY",
  "status_date": "2024-01-01",
  "maturity_date": "2024-12-31",
  "purchase_date": "2024-01-15",
  "price_at_purchase_date": 2200,
  "future_price": 2200,
  "contract_structure": "GOLD"
}

In [ ]:
!jactus simulate --type FUTUR --attrs /tmp/gold_future.json \
    --observer market --observer-params '{"risk_factors": {"GOLD": 2500}}'

### 1.7 List Available Observers (CLI)

In [ ]:
!jactus observer list

### 1.8 Portfolio Simulation (CLI)

Simulate a portfolio of mixed contract types from a single JSON file.

In [ ]:
%%writefile /tmp/demo_portfolio.json
{
  "portfolio_id": "DEMO-PORTFOLIO",
  "observer": { "type": "constant", "params": { "rate": 0.0 } },
  "contracts": [
    {
      "type": "PAM",
      "attrs": {
        "contract_id": "BOND-A",
        "contract_role": "RPA",
        "status_date": "2024-01-01",
        "initial_exchange_date": "2024-01-15",
        "maturity_date": "2027-01-15",
        "notional_principal": 100000,
        "nominal_interest_rate": 0.05,
        "interest_payment_cycle": "6M",
        "day_count_convention": "30E360"
      }
    },
    {
      "type": "ANN",
      "attrs": {
        "contract_id": "MORTGAGE-A",
        "contract_role": "RPA",
        "status_date": "2024-01-01",
        "initial_exchange_date": "2024-01-15",
        "maturity_date": "2029-01-15",
        "notional_principal": 200000,
        "nominal_interest_rate": 0.06,
        "interest_payment_cycle": "1M",
        "principal_redemption_cycle": "1M",
        "day_count_convention": "A360"
      }
    },
    {
      "type": "LAM",
      "attrs": {
        "contract_id": "AUTO-LOAN",
        "contract_role": "RPA",
        "status_date": "2024-01-01",
        "initial_exchange_date": "2024-01-15",
        "maturity_date": "2026-01-15",
        "notional_principal": 60000,
        "nominal_interest_rate": 0.05,
        "principal_redemption_cycle": "6M",
        "next_principal_redemption_amount": 15000,
        "principal_redemption_anchor": "2024-07-15",
        "interest_payment_cycle": "6M",
        "interest_calculation_base": "NT",
        "day_count_convention": "A360"
      }
    }
  ]
}

In [ ]:
!jactus portfolio simulate --file /tmp/demo_portfolio.json

### 1.9 JSON Output for Automation

Every CLI command supports `--output json` (a global flag) for piping into other tools.

In [ ]:
!jactus --output json simulate --type PAM --attrs /tmp/bond.json

---
## Part 2: Python API -- Same Contracts, Full Control

The Python API gives you programmatic access, visualization, and JAX integration.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import jactus
from jactus.contracts import create_contract
from jactus.core import (
    ActusDateTime,
    ContractAttributes,
    ContractRole,
    ContractType,
    DayCountConvention,
)
from jactus.observers import (
    ConstantRiskFactorObserver,
    DictRiskFactorObserver,
)

print(f"JACTUS version: {jactus.__version__}")

SD = ActusDateTime(2024, 1, 1)
IED = ActusDateTime(2024, 1, 15)


def show_events(result, max_events=10, title=None):
    """Compact event table for any contract type."""
    if title:
        print(f"\n{title}")
        print("=" * 65)
    print(f"{'Date':<12} {'Event':<6} {'Payoff':>14}  {'Notional':>14}")
    print("-" * 50)
    events = result.events
    shown = events[:max_events]
    for e in shown:
        t = e.event_time
        date_str = f"{t.year}-{t.month:02d}-{t.day:02d}"
        payoff = float(e.payoff)
        nt = float(e.state_post.nt) if e.state_post else 0.0
        payoff_s = f"${payoff:>12,.2f}" if payoff != 0 else f"{'--':>13}"
        print(f"{date_str:<12} {e.event_type.name:<6} {payoff_s}  ${nt:>12,.2f}")
    if len(events) > max_events:
        print(f"  ... ({len(events) - max_events} more events)")
    inflows = sum(float(e.payoff) for e in events if float(e.payoff) > 0)
    outflows = sum(float(e.payoff) for e in events if float(e.payoff) < 0)
    print(f"\nInflows: ${inflows:,.2f}  |  Outflows: ${outflows:,.2f}  |  Net: ${inflows + outflows:,.2f}")

### 2.1 PAM Bond -- Same Contract, Now in Python

In [ ]:
pam_attrs = ContractAttributes(
    contract_id="BOND-001", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2027, 1, 15),
    notional_principal=100_000.0, nominal_interest_rate=0.05,
    interest_payment_cycle="6M", day_count_convention="30E360",
    currency="USD",
)
pam_result = create_contract(pam_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(pam_result, title="PAM: $100K Bond at 5%, Semi-Annual, 3 Years")

### 2.2 ANN Mortgage -- Python with Visualization

In [ ]:
ann_attrs = ContractAttributes(
    contract_id="MORTGAGE-001", contract_type=ContractType.ANN,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2029, 1, 15),
    notional_principal=200_000.0, nominal_interest_rate=0.06,
    interest_payment_cycle="1M", principal_redemption_cycle="1M",
    day_count_convention="A360", currency="USD",
)
ann_result = create_contract(ann_attrs, ConstantRiskFactorObserver(0.06)).simulate()
show_events(ann_result, max_events=8, title="ANN: $200K Mortgage at 6%, Monthly, 5 Years")

In [ ]:
# Visualize principal + interest composition
pr_events = [e for e in ann_result.events if e.event_type.name == "PR"]
ip_events = [e for e in ann_result.events if e.event_type.name == "IP"]

# Pair up IP and PR events by matching dates
pr_by_date = {(e.event_time.year, e.event_time.month): float(e.payoff) for e in pr_events}
ip_by_date = {(e.event_time.year, e.event_time.month): float(e.payoff) for e in ip_events}
common_dates = sorted(set(pr_by_date.keys()) & set(ip_by_date.keys()))

pr_amounts = [pr_by_date[d] for d in common_dates]
ip_amounts = [ip_by_date[d] for d in common_dates]
labels = [f"{d[0]}-{d[1]:02d}" for d in common_dates]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: stacked bar of payment composition
x = range(len(common_dates))
ax1.bar(x, ip_amounts, label="Interest", color="gold", alpha=0.8)
ax1.bar(x, pr_amounts, bottom=ip_amounts, label="Principal", color="navy", alpha=0.8)
ax1.set_xlabel("Payment Number")
ax1.set_ylabel("Payment Amount ($)")
ax1.set_title("Mortgage Payment Composition")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: principal balance over time
balances = [float(e.state_post.nt) for e in ann_result.events if e.state_post]
ax2.plot(range(len(balances)), balances, color="crimson", linewidth=2)
ax2.fill_between(range(len(balances)), balances, alpha=0.15, color="crimson")
ax2.set_xlabel("Event Sequence")
ax2.set_ylabel("Outstanding Principal ($)")
ax2.set_title("Principal Balance Over Time")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 2.3 Interest Rate Swap (SWPPV) in Python

In [ ]:
swppv_attrs = ContractAttributes(
    contract_id="IRS-001", contract_type=ContractType.SWPPV,
    contract_role=ContractRole.RFL,
    status_date=SD, initial_exchange_date=IED,
    maturity_date=ActusDateTime(2027, 1, 15),
    currency="USD", notional_principal=1_000_000.0,
    nominal_interest_rate=0.03,
    nominal_interest_rate_2=0.04,
    day_count_convention=DayCountConvention.A360,
    interest_payment_cycle="6M",
)
swppv_result = create_contract(swppv_attrs, ConstantRiskFactorObserver(0.0)).simulate()
show_events(swppv_result, max_events=10, title="SWPPV: $1M IRS, Fixed 3% vs Float 4%, 3Y")

### 2.4 Options -- Call and Put with Payoff Diagram

In [ ]:
call_attrs = ContractAttributes(
    contract_id="CALL-100", contract_type=ContractType.OPTNS,
    contract_role=ContractRole.BUY, status_date=SD,
    maturity_date=ActusDateTime(2025, 1, 15), currency="USD",
    option_type="C", option_strike_1=100.0, option_exercise_type="E",
    purchase_date=ActusDateTime(2024, 1, 15), price_at_purchase_date=5.0,
    contract_structure="SPX",
)
spx_obs = DictRiskFactorObserver({"SPX": 115.0})
call_result = create_contract(call_attrs, spx_obs).simulate()
show_events(call_result, title="Call Option: Strike=$100, Spot=$115, Premium=$5")

put_attrs = ContractAttributes(
    contract_id="PUT-100", contract_type=ContractType.OPTNS,
    contract_role=ContractRole.BUY, status_date=SD,
    maturity_date=ActusDateTime(2025, 1, 15), currency="USD",
    option_type="P", option_strike_1=100.0, option_exercise_type="E",
    purchase_date=ActusDateTime(2024, 1, 15), price_at_purchase_date=3.0,
    contract_structure="SPX",
)
put_result = create_contract(put_attrs, spx_obs).simulate()
show_events(put_result, title="Put Option: Strike=$100, Spot=$115, Premium=$3")

In [ ]:
prices = np.linspace(70, 140, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

call_payoff = np.maximum(prices - 100, 0) - 5
ax1.plot(prices, call_payoff, "g-", lw=2)
ax1.axhline(0, color="gray", ls="-", alpha=0.3)
ax1.axvline(100, color="red", ls="--", alpha=0.5, label="Strike $100")
ax1.fill_between(prices, 0, call_payoff, where=call_payoff > 0, alpha=0.15, color="green")
ax1.set_title("Call Option P&L (Premium=$5)")
ax1.set_xlabel("Stock Price at Expiry ($)")
ax1.set_ylabel("P&L ($)")
ax1.legend()
ax1.grid(alpha=0.3)

put_payoff = np.maximum(100 - prices, 0) - 3
ax2.plot(prices, put_payoff, "r-", lw=2)
ax2.axhline(0, color="gray", ls="-", alpha=0.3)
ax2.axvline(100, color="green", ls="--", alpha=0.5, label="Strike $100")
ax2.fill_between(prices, 0, put_payoff, where=put_payoff > 0, alpha=0.15, color="red")
ax2.set_title("Put Option P&L (Premium=$3)")
ax2.set_xlabel("Stock Price at Expiry ($)")
ax2.set_ylabel("P&L ($)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 2.5 JAX Autodiff: Risk Metrics Without Bumping

In [ ]:
import jax

from jactus.contracts.pam_array import precompute_pam_arrays, simulate_pam_array

grad_attrs = ContractAttributes(
    contract_id="GRAD-001", contract_type=ContractType.PAM,
    contract_role=ContractRole.RPA,
    status_date=ActusDateTime(2025, 6, 1),
    initial_exchange_date=ActusDateTime(2022, 6, 1),
    maturity_date=ActusDateTime(2030, 6, 1),
    notional_principal=100_000.0, nominal_interest_rate=0.05,
    interest_payment_cycle="6M", day_count_convention=DayCountConvention.A360,
    currency="USD",
)
init_state, et, yf, rf, params = precompute_pam_arrays(
    grad_attrs, ConstantRiskFactorObserver(0.0)
)
_, sim_payoffs = simulate_pam_array(init_state, et, yf, rf, params)
cum_yf = jnp.cumsum(yf)


def pv_of_yield(discount_rate):
    """Present value as a differentiable function of the discount rate."""
    disc = jnp.power(1.0 + discount_rate, -cum_yf)
    return jnp.sum(sim_payoffs * disc)


base_yield = jnp.float32(0.045)
pv_val = float(pv_of_yield(base_yield))
dpv_dy = float(jax.grad(pv_of_yield)(base_yield))
d2pv_dy2 = float(jax.grad(jax.grad(pv_of_yield))(base_yield))

dv01 = dpv_dy * 0.0001
mod_duration = -dpv_dy / pv_val
convexity = d2pv_dy2 / pv_val

print("Risk Metrics via JAX Automatic Differentiation")
print("=" * 55)
print(f"Coupon:            {grad_attrs.nominal_interest_rate:.2%}")
print(f"Discount rate:     {float(base_yield):.2%}")
print(f"PV:                ${pv_val:,.2f}")
print(f"DV01:              ${dv01:,.4f}")
print(f"Modified Duration: {mod_duration:.4f}")
print(f"Convexity:         {convexity:.4f}")
print("=" * 55)

---
## Part 3: Behavioral Risk -- PrepaymentSurfaceObserver

JACTUS supports **behavioral risk models** that inject callout events into the simulation timeline. The `PrepaymentSurfaceObserver` uses a 2D surface indexed by:

- **Spread** = contract rate - market rate (x-axis)
- **Loan age** = years since inception (y-axis)

Higher spreads and seasoned loans tend to prepay more.

### 3.1 Define the Prepayment Surface

In [ ]:
from jactus.observers.prepayment import PrepaymentSurfaceObserver
from jactus.observers.scenario import Scenario
from jactus.utilities.surface import Surface2D

# Prepayment surface: spread (%) x loan age (years) -> annual prepayment rate
spreads = jnp.array([-2.0, -1.0, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
ages = jnp.array([0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])

# Realistic prepayment surface: rates peak at high spread + moderate age
prepay_values = jnp.array([
    # age: 0.0   0.5   1.0   2.0   3.0   5.0   7.0  10.0
    [0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00, 0.00],  # spread=-2%
    [0.00, 0.00, 0.01, 0.01, 0.01, 0.00, 0.00, 0.00],  # spread=-1%
    [0.00, 0.01, 0.02, 0.02, 0.02, 0.01, 0.01, 0.00],  # spread= 0%
    [0.00, 0.01, 0.03, 0.04, 0.03, 0.02, 0.01, 0.00],  # spread= 0.5%
    [0.00, 0.02, 0.05, 0.07, 0.06, 0.04, 0.02, 0.01],  # spread= 1%
    [0.01, 0.03, 0.08, 0.12, 0.10, 0.06, 0.03, 0.01],  # spread= 1.5%
    [0.01, 0.05, 0.12, 0.18, 0.15, 0.09, 0.05, 0.02],  # spread= 2%
    [0.02, 0.07, 0.16, 0.24, 0.20, 0.12, 0.06, 0.03],  # spread= 2.5%
    [0.03, 0.10, 0.22, 0.30, 0.25, 0.15, 0.08, 0.04],  # spread= 3%
])

surface = Surface2D(
    x_margins=spreads,
    y_margins=ages,
    values=prepay_values,
)

print(f"Surface shape: {prepay_values.shape[0]} spreads x {prepay_values.shape[1]} ages")
print(f"Spread range:  [{float(spreads[0]):.1f}%, {float(spreads[-1]):.1f}%]")
print(f"Age range:     [{float(ages[0]):.1f}, {float(ages[-1]):.1f}] years")
print(f"Max prepay:    {float(prepay_values.max()):.0%}")

### 3.2 Visualize the 2D Prepayment Surface

A heatmap shows where prepayments concentrate: high spread, seasoned loans.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Heatmap ---
im = ax1.imshow(
    np.array(prepay_values),
    aspect="auto", origin="lower", cmap="YlOrRd",
    extent=[float(ages[0]), float(ages[-1]), float(spreads[0]), float(spreads[-1])],
)
ax1.set_xlabel("Loan Age (Years)", fontsize=12)
ax1.set_ylabel("Spread: Contract Rate - Market Rate (%)", fontsize=12)
ax1.set_title("Prepayment Surface (Annual Rate)", fontsize=14)
cb = fig.colorbar(im, ax=ax1, label="Prepayment Rate", format="%.0%%")

# --- Right: 3D Surface ---
ax2.remove()
ax2 = fig.add_subplot(1, 2, 2, projection="3d")

# Create fine mesh for interpolation
fine_spreads = np.linspace(float(spreads[0]), float(spreads[-1]), 50)
fine_ages = np.linspace(float(ages[0]), float(ages[-1]), 50)
X, Y = np.meshgrid(fine_ages, fine_spreads)

Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = float(surface.evaluate(Y[i, j], X[i, j]))

surf = ax2.plot_surface(X, Y, Z, cmap="YlOrRd", alpha=0.9, edgecolor="none")
ax2.set_xlabel("Loan Age (Years)")
ax2.set_ylabel("Spread (%)")
ax2.set_zlabel("Prepay Rate")
ax2.set_title("3D Prepayment Surface", fontsize=14)
ax2.view_init(elev=25, azim=-60)

plt.tight_layout()
plt.show()

### 3.3 Create the Prepayment Observer and Scenario

In [ ]:
prepay_observer = PrepaymentSurfaceObserver(
    surface=surface,
    fixed_market_rate=0.04,
    prepayment_cycle="6M",
    model_id="demo-prepayment",
)

scenario = Scenario(
    scenario_id="prepay-demo",
    description="Base case with prepayment behavior at 4% market rate",
    market_observers={"rates": ConstantRiskFactorObserver(0.04)},
    behavior_observers={"prepayment": prepay_observer},
)

print(f"Scenario:     {scenario.scenario_id}")
print(f"Description:  {scenario.description}")
print(f"Market obs:   {list(scenario.market_observers.keys())}")
print(f"Behavioral:   {list(scenario.behavior_observers.keys())}")

### 3.4 Simulate a Mortgage with Prepayment Behavior

The mortgage has a 6% coupon vs 4% market rate (2% spread), so significant prepayment is expected.

In [ ]:
prepay_attrs = ContractAttributes(
    contract_id="PPY-MORTGAGE", contract_type=ContractType.ANN,
    contract_role=ContractRole.RPA, status_date=SD,
    initial_exchange_date=IED, maturity_date=ActusDateTime(2034, 1, 15),
    notional_principal=300_000.0, nominal_interest_rate=0.06,
    interest_payment_cycle="1M", principal_redemption_cycle="1M",
    day_count_convention="A360", currency="USD",
)

# Generate callout events from the scenario
callout_events = scenario.get_callout_events(prepay_attrs)

print(f"Callout events generated: {len(callout_events)}")
print(f"\n{'Date':<14} {'Type':<6} {'Model ID'}")
print("-" * 45)
for ce in callout_events[:10]:
    print(f"{ce.time.year}-{ce.time.month:02d}-{ce.time.day:02d}   {ce.callout_type:<6} {ce.model_id}")
if len(callout_events) > 10:
    print(f"  ... ({len(callout_events) - 10} more)")

print(f"\nContract rate: {prepay_attrs.nominal_interest_rate:.0%}")
print(f"Market rate:   {0.04:.0%}")
print(f"Spread:        {prepay_attrs.nominal_interest_rate - 0.04:.0%} -> triggers prepayment")

### 3.5 Prepayment Rate by Loan Age

Querying the surface along the 2% spread line shows how prepayment evolves with loan age.

In [ ]:
# Query the surface at different spreads and ages
query_ages = np.linspace(0, 10, 100)
spread_levels = [-1.0, 0.0, 1.0, 2.0, 3.0]
colors = ["steelblue", "gray", "orange", "crimson", "darkred"]

fig, ax = plt.subplots(figsize=(10, 5))

for spread, color in zip(spread_levels, colors):
    rates = [float(surface.evaluate(spread, age)) for age in query_ages]
    ax.plot(query_ages, [r * 100 for r in rates], label=f"Spread={spread:+.0f}%",
            color=color, linewidth=2)

ax.set_xlabel("Loan Age (Years)", fontsize=12)
ax.set_ylabel("Annual Prepayment Rate (%)", fontsize=12)
ax.set_title("Prepayment Rate by Loan Age at Different Spread Levels", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

---
## Part 4: Performance -- Batch Simulation with JAX

JACTUS uses JIT-compiled `jax.lax.scan` kernels to simulate thousands of contracts simultaneously.

In [ ]:
import random
import time

from jactus.contracts.pam_array import batch_simulate_pam_auto, prepare_pam_batch

rng = random.Random(42)
batch_contracts = []
obs = ConstantRiskFactorObserver(0.0)
for i in range(1200):
    orig_year = rng.randint(2020, 2024)
    orig_month = rng.randint(1, 12)
    term = rng.randint(3, 10)
    mat = ActusDateTime(orig_year + term, orig_month, 15)
    if mat <= ActusDateTime(2025, 6, 1):
        continue
    a = ContractAttributes(
        contract_id=f"LOAN-{i:04d}", contract_type=ContractType.PAM,
        contract_role=ContractRole.RPA,
        status_date=ActusDateTime(2025, 6, 1),
        initial_exchange_date=ActusDateTime(orig_year, orig_month, 15),
        maturity_date=mat,
        notional_principal=round(rng.uniform(50_000, 500_000), -3),
        nominal_interest_rate=round(rng.uniform(0.03, 0.08), 4),
        day_count_convention=rng.choice([DayCountConvention.A360, DayCountConvention.A365]),
        interest_payment_cycle=rng.choice(["1M", "3M", "6M"]),
    )
    batch_contracts.append((a, obs))

# Prepare arrays
t0 = time.perf_counter()
states, et, yf, rf, params, masks = prepare_pam_batch(batch_contracts)
t_prep = time.perf_counter() - t0

# JIT warm-up
final_states, payoffs = batch_simulate_pam_auto(states, et, yf, rf, params)
payoffs.block_until_ready()

# Timed run
t0 = time.perf_counter()
final_states, payoffs = batch_simulate_pam_auto(states, et, yf, rf, params)
payoffs.block_until_ready()
t_kernel = time.perf_counter() - t0

print(f"Batch Simulation: {len(batch_contracts):,} PAM Loans")
print("=" * 50)
print(f"Pre-computation:  {t_prep * 1000:.1f} ms")
print(f"JIT kernel:       {t_kernel * 1000:.3f} ms")
print(f"Total:            {(t_prep + t_kernel) * 1000:.1f} ms")
print(f"Throughput:       {len(batch_contracts) / (t_prep + t_kernel):,.0f} contracts/sec")
print(f"Backend:          {jax.default_backend()}")

In [ ]:
# Portfolio PV sensitivity across 200 yield scenarios
masked_payoffs = payoffs * masks
cum_yfs = jnp.cumsum(yf, axis=1)

rates = jnp.linspace(0.01, 0.10, 200)
pvs = []
for rate in rates:
    disc = jnp.power(1.0 + float(rate), -cum_yfs)
    pvs.append(float(jnp.sum(masked_payoffs * disc)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([float(r) * 100 for r in rates], [pv / 1e6 for pv in pvs],
        linewidth=2, color="navy")
ax.set_xlabel("Discount Rate (%)")
ax.set_ylabel("Portfolio PV ($M)")
ax.set_title(f"PV Sensitivity -- {len(batch_contracts):,} PAM Loans x 200 Scenarios")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Summary

| Feature | CLI | Python |
|---------|-----|--------|
| Discover contracts | `jactus contract list` | `ContractType` enum |
| Validate attributes | `jactus contract validate` | `ContractAttributes()` |
| Simulate | `jactus simulate` | `create_contract().simulate()` |
| Risk metrics | `jactus risk sensitivities` | `jax.grad()` on PV |
| Portfolio | `jactus portfolio simulate` | `simulate_portfolio()` |
| Observers | `jactus observer list` | `PrepaymentSurfaceObserver`, etc. |
| JSON output | `--output json` | `.to_dict()` on events |
| Batch perf | -- | `batch_simulate_pam_auto()` |

### Resources

- **Documentation**: [pedronahum.github.io/JACTUS](https://pedronahum.github.io/JACTUS/)
- **GitHub**: [github.com/pedronahum/JACTUS](https://github.com/pedronahum/JACTUS)
- **PyPI**: [pypi.org/project/jactus](https://pypi.org/project/jactus/)
- **ACTUS Standard**: [actusfrf.org](https://www.actusfrf.org/)

| Notebook | Topic |
|----------|-------|
| [00 - Getting Started](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/00_getting_started_pam.ipynb) | PAM in detail |
| [01 - Mortgage](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/01_annuity_mortgage.ipynb) | Mortgage amortization |
| [02 - Options](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/02_options_contracts.ipynb) | Options contracts |
| [03 - Interest Rate Cap](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/03_interest_rate_cap.ipynb) | Caps & floors |
| [04 - Stock & Commodity](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/04_stock_commodity.ipynb) | STK & COM |
| [05 - GPU/TPU Benchmark](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/05_gpu_tpu_portfolio_benchmark.ipynb) | Performance |
| [06 - Gallery](https://colab.research.google.com/github/pedronahum/JACTUS/blob/main/examples/notebooks/06_gallery_of_contracts.ipynb) | All 18 types |